# Score your own VQE result

You ran a VQE. You have an energy. **Is it any good?**

Answering that needs the exact ground state of the same active-space Hamiltonian, and
getting one means installing PySCF and waiting. This notebook skips that: QEncode ships
the references for all 16 suite molecules inside the package, so scoring costs one
function call and no chemistry stack.

In about two minutes you will know:

| | |
|---|---|
| **the gap** | how far your energy is from the exact answer for your own problem |
| **the tier** | chemical accuracy (1.6 mHa), the certification threshold (10 mHa), or research |
| **the margin** | how much room you have before the result stops meeting the threshold |
| **the risk** | whether your optimiser and ansatz make that margin fragile on another machine |
| **the rank** | where the number would sit among the published entries for the same molecule |

**What this is not.** It is not a certification — that requires the pipeline to generate
the entry with recorded provenance, a content hash and a signature, and the last section
shows how. It is not a hardware result: every reference here is an exact statevector
simulation. And the gap is measured against exact diagonalisation of *your own active
space*, not against experiment.

*Run top to bottom. The only cell you need to edit is Step 1.*

## Setup

```bash
pip install qencode-benchmark
```

Scoring itself imports **no chemistry stack — not even NumPy**: the references are a small
JSON table inside the package, so a score costs one file read. (The default install still
pulls the pipeline's pinned stack, because that is what `qencode run` needs. If you only
want to score, `pip install --no-deps qencode-benchmark` is enough — verified.)

Importing `qencode` also pins BLAS to a single thread, which is what makes a VQE
repeatable at all. The reason is in Step 6.

In [1]:
import qencode

print("qencode", qencode.__version__)
print("BLAS pinned to one thread:", qencode.threads_pinned())
print()
print("Scorable problems (%d):" % len(qencode.available()))
for mol, basis, orb in qencode.available():
    print("   %-12s %-9s %s orbitals" % (mol, basis, orb))

qencode 4.5.0
BLAS pinned to one thread: True

Scorable problems (16):
   BeH2         cc-pvdz   hf orbitals
   C4H4         cc-pvdz   casscf orbitals
   C4H6         cc-pvdz   hf orbitals
   H10          cc-pvdz   casscf orbitals
   H2           cc-pvdz   hf orbitals
   H2CO         cc-pvdz   hf orbitals
   H2O          cc-pvdz   hf orbitals
   H4           cc-pvdz   hf orbitals
   H6           cc-pvdz   casscf orbitals
   H8           cc-pvdz   casscf orbitals
   HF           cc-pvdz   hf orbitals
   LiH          cc-pvdz   hf orbitals
   N2           cc-pvdz   casscf orbitals
   NH3          cc-pvdz   hf orbitals
   benzene      cc-pvdz   casscf orbitals
   water_dimer  cc-pvdz   hf orbitals


## Step 1 — your result

Edit this cell. The only value you cannot guess is your energy; everything else describes
the problem you solved.

**The problem must match.** A gap measured against a different active space, geometry or
basis is not a worse number, it is a meaningless one — so declare your active space and
let the check catch a mismatch. Step 2 prints exactly what you have to have run.

`MY_OPTIMIZER` and `MY_ANSATZ` are optional, but without them you lose the fragility
assessment, which is the part you cannot get anywhere else.

In [2]:
# ── EDIT ME ──────────────────────────────────────────────────────────────────
MY_ENERGY       = -7.9835        # Hartree, your best variational energy
MY_MOLECULE     = "LiH"
MY_ACTIVE_SPACE = (4, 4)         # (electrons, spatial orbitals) -- checked, not assumed
MY_OPTIMIZER    = "COBYLA"       # or "L-BFGS-B", "Adam", "SPSA", None
MY_ANSATZ       = "hea"          # or "uccsd", "adapt", None
# ─────────────────────────────────────────────────────────────────────────────

## Step 2 — the problem you have to have solved

These are the exact inputs behind the reference energy. If your geometry, charge, spin,
basis or active space differs from this, stop here: the comparison will not mean anything.
(Distances are in Ångström.)

In [3]:
ref = qencode.reference(MY_MOLECULE)

for k in ("molecule", "basis", "geometry", "charge", "spin",
          "active_electrons", "active_orbitals", "orbital_optimization"):
    print("%-22s %s" % (k, ref[k]))

print()
print("%-22s %.10f Ha   <- what you are scored against"
      % ("exact ground state", ref["exact_qubit_ground_energy_hartree"]))
print("%-22s %.10f Ha" % ("Hartree-Fock", ref["hf_energy_hartree"]))
print("%-22s %.10f Ha" % ("CCSD(T)", ref["ccsd_t_energy_hartree"]))
print("%-22s %d" % ("published entries", ref["n_published_entries"]))

molecule               LiH
basis                  cc-pvdz
geometry               Li 0.0 0.0 0.0; H 0.0 0.0 1.6
charge                 0
spin                   0
active_electrons       4
active_orbitals        4
orbital_optimization   hf

exact ground state     -7.9837729770 Ha   <- what you are scored against
Hartree-Fock           -7.9836422316 Ha
CCSD(T)                -8.0147469479 Ha
published entries      3


## Step 3 — the score

One call. It raises rather than guessing if the active space does not match, and it checks
the variational principle before it reports any gap — an energy *below* the exact ground
state is not a good result, it means the problem you solved is not the one you think.

In [4]:
s = qencode.score(
    MY_ENERGY,
    molecule=MY_MOLECULE,
    active_space=MY_ACTIVE_SPACE,
    optimizer=MY_OPTIMIZER,
    ansatz=MY_ANSATZ,
)

print(s.report())

QEncode score -- LiH / cc-pvdz / hf orbitals
  your energy             -7.9835000000 Ha
  exact ground state      -7.9837729770 Ha   (CASCI in the declared active space)
  gap                      0.0002729770 Ha   = 0.273 mHa

  reaches CHEMICAL ACCURACY (< 1.6 mHa) and would meet the 10 mHa certification threshold
  margin             9.727e-03 Ha (97.3% of the threshold)

  optimiser          COBYLA (gradient-free)
  amplifying         YES -- gradient-free optimiser on an unstructured
                     ansatz. Re-run elsewhere, energies in this class
                     have moved by up to 1e-2 Ha. If the margin above
                     is thin, treat it as provisional.

  among published    #3 of 4 QEncode entries for this problem
  best published gap 0.003 mHa

  Read before quoting this:
    - The gap is measured against the exact ground state of the SAME active-space Hamiltonian, not against experiment and not against a complete-basis limit. It isolates the algorithm's err

## Step 4 — is your margin safe?

The gap alone does not tell you whether the result will survive being re-run somewhere
else. Two things decide that.

**Margin** is `10 mHa − gap`: how far the energy can move before the result stops meeting
the threshold. Below 20% of the threshold is thin.

**Amplification** is whether it *will* move. A gradient-free optimiser picks its next step
by comparing two nearly equal energies, so a difference in the thirteenth decimal — a
different BLAS, a different NumPy — can flip a comparison and send the run into a
different local minimum. But the optimiser alone is not the rule. Measured on H₄, holding
molecule, basis, mapping and environment fixed and changing **only the ansatz**:

| ansatz | optimiser | energy moved across environments |
|---|---|---|
| ADAPT | COBYLA *inner* | 3.4 × 10⁻⁸ Ha |
| HEA | plain COBYLA | 8.8 × 10⁻⁴ Ha |

**25,595× apart.** ADAPT selects its operators by analytic gradient, so its structure is
gradient-determined and the gradient-free optimiser only polishes a small, well-conditioned
set. An unstructured ansatz hands the same optimiser a landscape full of near-degenerate
minima. So the risk is the *conjunction*: gradient-free **and** unstructured.

If your run is amplifying and your margin is thin, treat the result as provisional.

In [5]:
print("margin        ", "n/a" if s.margin_ha is None
      else "%.3e Ha  (%.1f%% of the threshold)%s"
           % (s.margin_ha, 100 * s.margin_fraction, "  THIN" if s.thin_margin else ""))
print("optimiser     ", s.optimizer, "->", s.optimiser_family)
print("amplifying    ", {True: "YES", False: "no", None: "unknown"}[s.amplifies])
print()

verdict = ("cannot assess -- pass optimizer= and ansatz=" if s.amplifies is None else
           "provisional: thin margin AND an amplifying configuration"
           if (s.thin_margin and s.amplifies) else
           "amplifying, but the margin is wide enough to absorb it"
           if s.amplifies else
           "stable: this combination moves <= 1e-6 Ha across environments")
print("verdict:", verdict)

margin         9.727e-03 Ha  (97.3% of the threshold)
optimiser      COBYLA -> gradient-free
amplifying     YES

verdict: amplifying, but the margin is wide enough to absorb it


## Step 5 — how it compares

Every published QEncode entry for the same problem, with your result placed among them.
Gaps are only comparable *within* a molecule — never across molecules.

In [6]:
rows = [e for e in qencode.scoring.references_table()["entries"]
        if e["molecule"] == ref["molecule"]
        and e["orbital_optimization"] == ref["orbital_optimization"]]
rows.append({"mapping": "-- YOURS --", "ansatz": MY_ANSATZ or "?",
             "optimizer": MY_OPTIMIZER or "?", "gap_ha": s.gap_ha, "entry_id": None})
rows.sort(key=lambda r: r["gap_ha"])

print("%-4s %-16s %-16s %-26s %10s" % ("#", "mapping", "ansatz", "optimiser", "gap (mHa)"))
print("-" * 78)
for i, r in enumerate(rows, 1):
    mark = "  <<<" if r["entry_id"] is None else ""
    print("%-4d %-16s %-16s %-26s %10.3f%s"
          % (i, r["mapping"], r["ansatz"], str(r["optimizer"])[:26],
             r["gap_ha"] * 1e3, mark))

#    mapping          ansatz           optimiser                   gap (mHa)
------------------------------------------------------------------------------
1    jordan_wigner    uccsd_tapered    COBYLA                          0.003
2    jordan_wigner    hea              COBYLA                          0.096
3    -- YOURS --      hea              COBYLA                          0.273  <<<
4    parity           hea              COBYLA                          5.181


## Step 6 — is your *setup* reproducible?

A good energy that nobody can reproduce is not a result. Four things have to be true, and
only the first is about arithmetic:

1. **Deterministic arithmetic.** Threaded BLAS sums floating point in whatever order the
   cores finish. Setting a random seed does **not** fix this — the non-determinism is in
   the arithmetic, not the RNG. We found it in our own published numbers: the same LiH
   command returned 8.99 mHa or 0.53 mHa on identical hardware.
2. **Recorded package versions.**
3. **A recorded seed.**
4. **A recorded code version** — a clean commit.

Importing `qencode` handles #1 for you, but only if it happens *before* NumPy is imported.
The cell below checks that. The full four-check scorecard is a standalone script needing
only NumPy and SciPy — `qencode check` from a checkout, or
[`tools/check_vqe_reproducibility.py`](https://github.com/qencode-benchmark/qencode-benchmark/blob/master/tools/check_vqe_reproducibility.py)
run against your own project.

In [7]:
import os

pinned = qencode.threads_pinned()
print("BLAS threads pinned to 1:", pinned)
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    print("   %-24s %s" % (v, os.environ.get(v, "<unset>")))

if not pinned:
    print()
    print("NumPy was imported before qencode, so the thread pool is already built.")
    print("Restart the kernel and import qencode first.")

BLAS threads pinned to 1: True
   OMP_NUM_THREADS          1
   OPENBLAS_NUM_THREADS     1
   MKL_NUM_THREADS          1


## What to do next

**If the score looks good**, the honest next step is to stop self-reporting it. Generate a
real entry: same procedure, but with the environment pinned, the provenance recorded, and
a content hash over the result.

```bash
pip install qencode-benchmark
qencode run --molecule LiH --mapping jordan_wigner --ansatz-type uccsd
```

That writes a JSON entry you can verify from a clean checkout:

```bash
python scripts/verify_entry.py <entry>.json
```

**If your molecule is not in the 16**, QEncode has no reference for it and this notebook
cannot score it. Generating one needs the chemistry stack — `qencode run` computes the
CASCI reference as part of the pipeline.

**If you want it on the leaderboard**, see
[docs/SUBMISSIONS.md](https://github.com/qencode-benchmark/qencode-benchmark/blob/master/docs/SUBMISSIONS.md).
Certified and research-tier entries are both published; nothing is discarded for missing a
threshold.

---

- Leaderboard — <https://www.qencode-benchmark.org/leaderboard>
- What the numbers mean — <https://www.qencode-benchmark.org/leaderboard/guide>
- Why the threading bug mattered — <https://www.qencode-benchmark.org/blog/vqe-reproducibility-threading-bug>